In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS air_quality.reference;


In [0]:
%sql
CREATE TABLE IF NOT EXISTS air_quality.reference.city_lookup (
    latitude DOUBLE,
    longitude DOUBLE,
    city STRING,
    state STRING,
    country STRING,
    created_at TIMESTAMP
)
USING DELTA;


In [0]:
%pip install geopy

In [0]:
from geopy.geocoders import Nominatim
from datetime import datetime
import time


In [0]:
raw_locations = spark.read.json(
    "/Volumes/air_quality/openaq/ingestion/locations/"
)
coords = (
    raw_locations
    .selectExpr("explode(results) as r")
    .selectExpr(
        "r.coordinates.latitude",
        "r.coordinates.longitude"
    )
    .where("latitude IS NOT NULL AND longitude IS NOT NULL")
    .distinct()
)
# coords.show()

#Existing coordinates in the look up table.
existing_coords = (
    spark.table("air_quality.reference.city_lookup")
    .select("latitude", "longitude")
)

new_coords = (
    coords
    .join(
        existing_coords,
        (coords.latitude == existing_coords.latitude) &
        (coords.longitude == existing_coords.longitude),
        "left_anti"
    )
)

new_coords.show()

In [0]:
# Reverse Geocode.

geolocator = Nominatim(user_agent="openaq-databricks-project")

def reverse_geocode(lat, lon):
    try:
        location = geolocator.reverse((lat,lon), language= "en")
        if not location:
            return (None , None, None)
        
        address = location.raw['address']
        city = address.get('city')
        state = address.get('state')
        country = address.get('country')
        return city, state, country
    
    except Exception:
        return (None, None, None)

rows = []
for r in new_coords.collect():
    city, state, country = reverse_geocode(r.latitude, r.longitude)
    rows.append(
        (r.latitude, r.longitude, city, state, country, datetime.utcnow())
    )
    time.sleep(1)

# print(rows)

if rows:
    lookup_df =    spark.createDataFrame(rows, ["latitude", "longitude", "city", "state", "country", "created_at"]
                                         )
    lookup_df.write.mode("append").saveAsTable("air_quality.reference.city_lookup")

In [0]:
spark.read.table("air_quality.reference.city_lookup").show()